# 🔗 HPVD × Knowledge Layer — Integration Demo

Notebook ini mendemonstrasikan alur **end-to-end** HPVD mengambil data dari Knowledge Layer (KL) yang live,
menjalankan pipeline retrieval, dan menghasilkan **J14 (RetrievalRaw)**, **J15 (PhaseFilteredSet)**, dan **J16 (AnalogFamilyAssignment)**.

**Update:** Menggunakan KL versi terbaru dengan gap-resolution:
- ✅ **G1** — Snapshot pinning & versioning (`GET /snapshots/{id}`)
- ✅ **G2** — KL-managed chunks per document version (`GET .../chunks`)
- ✅ **G3** — Structured metadata (`domain`, `action_class`, `phase_label`, `tags`)
- ✅ **G6** — Search endpoint with metadata filters (`POST /documents/search`)
- ✅ Auth via `X-API-Key` header (no more positional `tenant_id` in calls)

---

**Pipeline:**
```
KL API (live) → KLDocumentLoader → DocumentChunk[] → HPVD Pipeline → J14 → J15 → J16
```


## 1. Connect to Knowledge Layer

In [ ]:
import os
from src.hpvd.adapters.kl_client import KLClient

KL_URL = "https://knowledge-layer-production.up.railway.app"

# Read tenant API key from environment.
# Expected for retrieval endpoints: kl_...
KL_API_KEY = os.environ.get("KL_API_KEY", "").strip()

if not KL_API_KEY:
    raise RuntimeError(
        "KL_API_KEY kosong. Set dulu env var KL_API_KEY (kl_...), lalu restart kernel dan run ulang Cell 3."
    )

client = KLClient(base_url=KL_URL, api_key=KL_API_KEY)

# Health endpoint is public, this checks service reachability only.
health = client.health_check()
print(f"✅ KL API reachable: {health}")

# Auth probe to protected endpoint so failures are visible earlier.
try:
    _probe = client.list_documents(limit=1)
    print(f"✅ Auth OK. /documents accessible (sample={len(_probe)})")
except Exception as exc:
    print("❌ Auth failed for /documents.")
    print("   Pastikan KL_API_KEY adalah tenant key (prefix kl_), bukan admin key (kla_).")
    print(f"   Detail: {type(exc).__name__}: {exc}")
    raise


✅ KL API reachable: {'message': 'Knowledge Layer API'}
✅ Auth OK. /documents accessible (sample=1)


### 1.1 Optional Bootstrap Tenant (for Observability)

Gunakan section ini jika ingin membuat tenant baru + API key langsung dari notebook.

Requirements:
- Set env var `KLA_ADMIN_KEY` (admin key `kla_...`)
- Proses ini bersifat operasional, jalankan hanya saat bootstrap


In [ ]:
# Optional: bootstrap tenant + create tenant API key (kl_...) with observability logs
import os
from datetime import datetime
import pandas as pd

ADMIN_KEY = os.environ.get("KLA_ADMIN_KEY", "").strip()
TARGET_TENANT_ID = os.environ.get("KL_NEW_TENANT_ID", f"DEMO_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
TARGET_TENANT_NAME = os.environ.get("KL_NEW_TENANT_NAME", "HPVD Notebook Demo Tenant")
TARGET_KEY_NAME = os.environ.get("KL_NEW_KEY_NAME", "notebook-bootstrap-key")

if not ADMIN_KEY:
    print("⏭️ Skip bootstrap: set KLA_ADMIN_KEY if you want to create tenant/API key.")
else:
    admin_client = KLClient(base_url=KL_URL, admin_key=ADMIN_KEY)
    obs = []

    tenants = admin_client.list_tenants()
    obs.append({"step": "list_tenants", "ok": True, "count": len(tenants), "detail": "Fetched tenant list"})

    existing = next((t for t in tenants if t.id == TARGET_TENANT_ID), None)

    if existing is None:
        created = admin_client.create_tenant(name=TARGET_TENANT_NAME, tenant_id=TARGET_TENANT_ID)
        obs.append({"step": "create_tenant", "ok": True, "count": 1, "detail": f"Created tenant {created.id}"})
    else:
        obs.append({"step": "create_tenant", "ok": True, "count": 0, "detail": f"Tenant already exists: {existing.id}"})

    key_created = admin_client.create_api_key(tenant_id=TARGET_TENANT_ID, name=TARGET_KEY_NAME)
    tenant_api_key = key_created.raw_key
    masked = tenant_api_key[:10] + "..." + tenant_api_key[-6:]
    obs.append({"step": "create_api_key", "ok": True, "count": 1, "detail": f"Created key {key_created.id} ({key_created.key_prefix})"})

    probe_client = KLClient(base_url=KL_URL, api_key=tenant_api_key)
    _docs = probe_client.list_documents(limit=1)
    obs.append({"step": "probe_documents", "ok": True, "count": len(_docs), "detail": "New tenant key can access /documents"})

    # Make this notebook session immediately usable without kernel restart.
    os.environ["KL_API_KEY"] = tenant_api_key
    client = probe_client

    print(f"✅ Bootstrap complete for tenant: {TARGET_TENANT_ID}")
    print(f"🔑 New KL_API_KEY (masked): {masked}")
    print("✅ KL_API_KEY injected into current kernel session.")
    print("➡️ Lanjut run Cell 6 (Fetch Documents).")
    display(pd.DataFrame(obs))


✅ Bootstrap complete for tenant: DEMO_20260314_202549
🔑 New KL_API_KEY (masked): kl_037b6f7...2d46f8
✅ KL_API_KEY injected into current kernel session.
➡️ Lanjut run Cell 6 (Fetch Documents).


,step,ok,count,detail
0,list_tenants,True,4,Fetched tenant list
1,create_tenant,True,1,Created tenant DEMO_20260314_202549
2,create_api_key,True,1,Created key 7acb0dc6-0a87-49a5-8746-b2fba92994...
3,probe_documents,True,0,New tenant key can access /documents


In [16]:
# Seed demo documents into current tenant key if still empty
import os
import subprocess
import sys

tenant_key = os.environ.get("KL_API_KEY", "").strip()
if not tenant_key:
    raise RuntimeError("KL_API_KEY kosong. Jalankan bootstrap dulu atau set KL_API_KEY (kl_...).")

cmd = [
    sys.executable,
    "scripts/seed_kl_data.py",
    "--api-key", tenant_key,
    "--base-url", KL_URL,
]

print("🚀 Seeding documents from data/banking_docs ...")
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Seeding failed with exit code {result.returncode}")

# Refresh the client after seeding
client = KLClient(base_url=KL_URL, api_key=tenant_key)
probe_docs = client.list_documents(limit=5)
print(f"✅ Seed selesai. Sample documents now available: {len(probe_docs)} (limit=5)")


🚀 Seeding documents from data/banking_docs ...
Knowledge Layer Seed Script v2
  KL API    : https://knowledge-layer-production.up.railway.app
  Data      : d:\project\M22\HPVD-M22\data\banking_docs
  Cases     : 3 (1437613, 1440632, 3585136)
  Tenant    : BANKING_CORE
  Dry run   : False

[OK] KL API reachable: {'message': 'Knowledge Layer API'}

Case 1437613:
  [OK] Created document: 65e28732-3c48-4e9f-aefd-beffd77343db â€” [1437613] 1. intimazione_17-11-2025 12.47.20.pdf
       type=INTIMAZIONE action=DEBT_COLLECTION phase=EXECUTION_PHASE
  [ERR] Failed: 1. intimazione_17-11-2025 12.47.20.pdf â€” Server error '500 Internal Server Error' for url 'https://knowledge-layer-production.up.railway.app/documents/65e28732-3c48-4e9f-aefd-beffd77343db/versions?uploaded_by=seed_script_v2'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
  [OK] Created document: ebce6084-46f9-401a-992b-4ec97497e0e9 â€” [1437613] 10. no concordato_17-11-2025 12.50.00.pdf
   

### 1.2 Optional Seed Banking Documents (if tenant is empty)

Jika output `Total documents in KL: 0`, jalankan **code cell tepat di atas section ini** untuk seed dataset demo dari folder lokal `data/banking_docs`.


## 2. Fetch Documents from KL

In [3]:
# Tenant is resolved from KL_API_KEY (tenant key: kl_...)
from collections import Counter

try:
    documents = client.list_documents(limit=100)
except Exception as exc:
    print("❌ Gagal ambil /documents")
    print("   Cek: Cell 3 harus sukses 'Auth OK'.")
    print("   Jika baru bootstrap, jalankan bootstrap cell lalu ulangi cell ini.")
    print(f"   Detail: {type(exc).__name__}: {exc}")
    raise

print(f"📄 Total documents in KL: {len(documents)}")

if len(documents) == 0:
    print("⚠️ Tenant masih kosong. Jalankan seed code cell di section 1.2, lalu ulangi cell ini.")

print()

# Show summary by document type
type_counts = Counter(d.document_type for d in documents)
print("Document types:")
for doc_type, count in type_counts.most_common():
    print(f"  {doc_type:25s} -> {count} docs")

# Show structured metadata coverage (G3)
has_metadata = sum(1 for d in documents if d.metadata and d.metadata.domain)
print(f"\nStructured metadata coverage (G3): {has_metadata}/{len(documents)}")


📄 Total documents in KL: 36

Document types:
  ESITO_LETTER              -> 4 docs
  NO_PREJUDICE              -> 4 docs
  DELIBERA                  -> 3 docs
  OTHER                     -> 3 docs
  RATE_DECLARATION          -> 2 docs
  AMMORTAMENTO              -> 2 docs
  EROGAZIONE                -> 2 docs
  CONTRACT                  -> 2 docs
  CREDIT_SPECIFICATION      -> 2 docs
  PEC_RECEIPT               -> 2 docs
  CREDIT_REPORT             -> 2 docs
  NO_DEFAULT                -> 2 docs
  INTIMAZIONE               -> 2 docs
  IDENTITY_DOC              -> 2 docs
  RISK_DECLARATION          -> 1 docs
  COMPLIANCE                -> 1 docs

Structured metadata coverage (G3): 36/36


In [4]:
# Show first 10 documents — now includes structured metadata (G3)
import pandas as pd

df_docs = pd.DataFrame([
    {
        "id": d.id[:8] + "...",
        "title": d.title[:50],
        "type": d.document_type,
        "domain": d.metadata.domain if d.metadata else "",
        "action_class": (d.metadata.action_class or "")[:25] if d.metadata else "",
        "phase_label": (d.metadata.phase_label or "") if d.metadata else "",
        "created_at": d.created_at[:19] if d.created_at else "",
    }
    for d in documents[:10]
])
df_docs


,id,title,type,domain,action_class,phase_label,created_at
0,3378c760...,[1440632] LetteraEsitoRichiedente_1440632_29-0...,ESITO_LETTER,finance,OUTCOME_NOTIFICATION,COMPLETION_PHASE,2026-03-13T07:14:03
1,de17f179...,[1440632] LetteraEsitoBeneficiario_1440632_29-...,ESITO_LETTER,finance,OUTCOME_NOTIFICATION,COMPLETION_PHASE,2026-03-13T07:14:03
2,9aa922d0...,[1440632] 9. dichiarazione tasso_18-12-2025 17...,RATE_DECLARATION,finance,RISK_ASSESSMENT,ANALYSIS_PHASE,2026-03-13T07:14:03
3,3eeac8a4...,[1440632] 8. Piano di ammortamento_18-12-2025 ...,AMMORTAMENTO,finance,REPAYMENT_PLAN,PLANNING_PHASE,2026-03-13T07:14:03
4,e0d23be5...,[1440632] 7. Erogazione_18-12-2025 17.14.19.pdf,EROGAZIONE,finance,DISBURSEMENT,EXECUTION_PHASE,2026-03-13T07:14:03
5,1ceff825...,[1440632] 6. Contratto finanziamento_18-12-202...,CONTRACT,finance,TRADE_EXECUTION,EXECUTION_PHASE,2026-03-13T07:14:03
6,70b3de40...,[1440632] 5.1 data della delibera_18-12-2025 1...,DELIBERA,finance,CREDIT_APPROVAL,DECISION_PHASE,2026-03-13T07:14:03
7,cbabeca8...,[1440632] 5. delibera banca_18-12-2025 17.13.5...,DELIBERA,finance,CREDIT_APPROVAL,DECISION_PHASE,2026-03-13T07:14:03
8,8fb5a9c5...,[1440632] 4. all 4_18-12-2025 17.11.50.pdf,OTHER,finance,OTHER,UNKNOWN_PHASE,2026-03-13T07:14:03
9,2b2c0c57...,[1440632] 3. Precisazione del credito_18-12-20...,CREDIT_SPECIFICATION,finance,CREDIT_ANALYSIS,ANALYSIS_PHASE,2026-03-13T07:14:03


## 3. Load & Convert to HPVD DocumentChunks

Loader sekarang menggunakan **KL-managed chunks** (G2 resolved):
- Untuk setiap dokumen, loader fetches chunks yang disimpan di KL (`GET .../chunks`)
- Jika tidak ada chunks — fallback ke synthetic chunk dari title + metadata
- Chunk metadata sudah include `kl_chunk_id`, `checksum_sha256`, `domain`, `action_class`, dll.


In [5]:
from src.hpvd.adapters.kl_document_loader import KLDocumentLoader

loader = KLDocumentLoader(client)

# Mode 1: load all documents → fetch KL-managed chunks (G2)
# No tenant_id needed — resolved from API key
chunks = loader.load_as_chunks()

# Show KL-managed vs synthetic fallback
kl_managed = sum(1 for c in chunks if not c.metadata.get("synthetic_chunk"))
synthetic_fallback = len(chunks) - kl_managed

print(f"🧩 Total DocumentChunks  : {len(chunks)}")
print(f"   KL-managed chunks    : {kl_managed}  ← real content from KL (G2)")
print(f"   Synthetic (fallback) : {synthetic_fallback}  ← title-based, no KL chunks yet")
print()

# Show topic distribution
topic_counts = Counter(c.topic for c in chunks)
print("Topic distribution (mapped from document_type / metadata.domain):")
for topic, count in topic_counts.most_common():
    print(f"  {topic:25s} → {count} chunks")


🧩 Total DocumentChunks  : 36
   KL-managed chunks    : 0  ← real content from KL (G2)
   Synthetic (fallback) : 36  ← title-based, no KL chunks yet

Topic distribution (mapped from document_type / metadata.domain):
  finance                   → 36 chunks


In [42]:
# Show sample chunks — now includes KL chunk metadata (G2 + G3)
df_chunks = pd.DataFrame([
    {
        "chunk_id": c.chunk_id[:22] + "...",
        "topic": c.topic,
        "doc_type": c.doc_type,
        "kl_chunk_id": c.metadata.get("kl_chunk_id", "—"),
        "domain": c.metadata.get("domain", ""),
        "action_class": (c.metadata.get("action_class") or "")[:20],
        "has_checksum": bool(c.metadata.get("checksum_sha256")),
        "text_preview": c.text[:60],
    }
    for c in chunks[:10]
])
df_chunks


,chunk_id,topic,doc_type,kl_chunk_id,domain,action_class,has_checksum,text_preview
0,kl_3378c760-a29b-419c-...,finance,ESITO_LETTER,—,finance,OUTCOME_NOTIFICATION,False,[1440632] LetteraEsitoRichiedente_1440632_29-0...
1,kl_de17f179-0da8-43ad-...,finance,ESITO_LETTER,—,finance,OUTCOME_NOTIFICATION,False,[1440632] LetteraEsitoBeneficiario_1440632_29-...
2,kl_9aa922d0-1de4-43a2-...,finance,RATE_DECLARATION,—,finance,RISK_ASSESSMENT,False,[1440632] 9. dichiarazione tasso_18-12-2025 17...
3,kl_3eeac8a4-5ca0-4ada-...,finance,AMMORTAMENTO,—,finance,REPAYMENT_PLAN,False,[1440632] 8. Piano di ammortamento_18-12-2025 ...
4,kl_e0d23be5-99d9-4957-...,finance,EROGAZIONE,—,finance,DISBURSEMENT,False,[1440632] 7. Erogazione_18-12-2025 17.14.19.pd...
5,kl_1ceff825-1877-4522-...,finance,CONTRACT,—,finance,TRADE_EXECUTION,False,[1440632] 6. Contratto finanziamento_18-12-202...
6,kl_70b3de40-3dcb-4b5a-...,finance,DELIBERA,—,finance,CREDIT_APPROVAL,False,[1440632] 5.1 data della delibera_18-12-2025 1...
7,kl_cbabeca8-255c-434e-...,finance,DELIBERA,—,finance,CREDIT_APPROVAL,False,[1440632] 5. delibera banca_18-12-2025 17.13.5...
8,kl_8fb5a9c5-001f-417d-...,finance,OTHER,—,finance,OTHER,False,[1440632] 4. all 4_18-12-2025 17.11.50.pdf. Do...
9,kl_2b2c0c57-e535-4f81-...,finance,CREDIT_SPECIFICATION,—,finance,CREDIT_ANALYSIS,False,[1440632] 3. Precisazione del credito_18-12-20...


In [36]:
# Diagnostic: which documents already have real KL chunks vs synthetic fallback?
versions_by_doc = {}
for doc in documents:
    try:
        versions = client.list_versions(doc.id)
        latest_version = max((v.version_number for v in versions), default=None)
        chunk_count = 0
        if latest_version is not None:
            try:
                chunk_count = len(client.list_chunks(doc.id, latest_version))
            except Exception:
                chunk_count = 0
        versions_by_doc[doc.id] = {
            "title": doc.title[:55],
            "document_type": doc.document_type,
            "latest_version": latest_version,
            "kl_chunk_count": chunk_count,
            "uses_fallback": chunk_count == 0,
        }
    except Exception:
        versions_by_doc[doc.id] = {
            "title": doc.title[:55],
            "document_type": doc.document_type,
            "latest_version": None,
            "kl_chunk_count": 0,
            "uses_fallback": True,
        }

df_chunk_diag = pd.DataFrame(list(versions_by_doc.values()))
print(f"📊 Docs with real KL chunks : {(~df_chunk_diag['uses_fallback']).sum()}")
print(f"📊 Docs using fallback      : {df_chunk_diag['uses_fallback'].sum()}")
print()
display(df_chunk_diag.sort_values(['uses_fallback', 'kl_chunk_count'], ascending=[False, False]).head(15))


📊 Docs with real KL chunks : 0
📊 Docs using fallback      : 36



,title,document_type,latest_version,kl_chunk_count,uses_fallback
0,[1440632] LetteraEsitoRichiedente_1440632_29-0...,ESITO_LETTER,None,0,True
1,[1440632] LetteraEsitoBeneficiario_1440632_29-...,ESITO_LETTER,None,0,True
2,[1440632] 9. dichiarazione tasso_18-12-2025 17...,RATE_DECLARATION,None,0,True
3,[1440632] 8. Piano di ammortamento_18-12-2025 ...,AMMORTAMENTO,None,0,True
4,[1440632] 7. Erogazione_18-12-2025 17.14.19.pdf,EROGAZIONE,None,0,True
5,[1440632] 6. Contratto finanziamento_18-12-202...,CONTRACT,None,0,True
6,[1440632] 5.1 data della delibera_18-12-2025 1...,DELIBERA,None,0,True
7,[1440632] 5. delibera banca_18-12-2025 17.13.5...,DELIBERA,None,0,True
8,[1440632] 4. all 4_18-12-2025 17.11.50.pdf,OTHER,None,0,True
9,[1440632] 3. Precisazione del credito_18-12-20...,CREDIT_SPECIFICATION,None,0,True


### 3.0 Async Chunk Polling Test (Delay vs Not Triggered)

Cell ini mengecek satu dokumen terbaru dan polling endpoint chunks beberapa kali.
Jika count tetap 0 sepanjang polling, indikasi kuat chunking tidak dipicu otomatis oleh upload flow saat ini.


In [7]:
import time
import pandas as pd

# Ensure prerequisites exist after kernel restart.
if "documents" not in globals():
    try:
        documents = client.list_documents(limit=100)
    except Exception as exc:
        raise RuntimeError(
            f"Cannot fetch documents for polling test: {type(exc).__name__}: {exc}"
        )

if not documents:
    raise RuntimeError("No documents available. Run fetch/seed flow first.")

# Scan latest documents and check whether they have versions.
docs_sorted = sorted(
    documents,
    key=lambda d: d.created_at or "",
    reverse=True,
 )

scan_rows = []
target_doc = None
target_versions = []

for d in docs_sorted[:25]:
    try:
        versions = client.list_versions(d.id)
        latest_version = max((v.version_number for v in versions), default=None)
        scan_rows.append(
            {
                "doc_id": d.id,
                "title": (d.title or "")[:70],
                "created_at": d.created_at or "",
                "version_count": len(versions),
                "latest_version": latest_version,
            }
        )
        if target_doc is None and versions:
            target_doc = d
            target_versions = versions
    except Exception as exc:
        scan_rows.append(
            {
                "doc_id": d.id,
                "title": (d.title or "")[:70],
                "created_at": d.created_at or "",
                "version_count": -1,
                "latest_version": None,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

df_scan = pd.DataFrame(scan_rows)
docs_with_versions = int((df_scan["version_count"] > 0).sum())

print(f"📄 Scanned docs: {len(df_scan)}")
print(f"📌 Docs with versions: {docs_with_versions}")

if docs_with_versions == 0:
    print("\n⚠️ Tidak ada dokumen dengan version di tenant ini.")
    print("   Upload/seed dokumen melalui flow yang membuat version, lalu jalankan ulang cell ini.")
    print("   Kamu bisa lihat ringkasan dokumen di bawah untuk verifikasi cepat.")
    display(df_scan[["doc_id", "title", "created_at", "version_count", "latest_version"]].head(15))
else:
    latest_version = max(target_versions, key=lambda v: v.version_number)
    doc_label = (target_doc.title or "")[:70]

    print("\n🔎 Polling chunks for latest document WITH version:")
    print(f"   doc_id         : {target_doc.id}")
    print(f"   title          : {doc_label}")
    print(f"   latest_version : {latest_version.version_number}")
    print()

    poll_seconds = 10
    poll_attempts = 12  # total 2 minutes
    chunk_counts = []

    for i in range(1, poll_attempts + 1):
        try:
            chunks_now = client.list_chunks(target_doc.id, latest_version.version_number)
            cnt = len(chunks_now)
        except Exception as exc:
            cnt = -1
            print(f"[{i:02d}/{poll_attempts}] list_chunks error: {type(exc).__name__}: {exc}")

        chunk_counts.append(cnt)
        ts = time.strftime("%H:%M:%S")
        if cnt >= 0:
            print(f"[{i:02d}/{poll_attempts}] {ts}  chunk_count={cnt}")
        else:
            print(f"[{i:02d}/{poll_attempts}] {ts}  chunk_count=ERR")

        if i < poll_attempts:
            time.sleep(poll_seconds)

    print("\nSummary:")
    print(f"Counts: {chunk_counts}")

    if any(c > 0 for c in chunk_counts if c >= 0):
        print("✅ Chunks appeared during polling -> asynchronous chunking likely active.")
    elif all(c == 0 for c in chunk_counts if c >= 0):
        print("⚠️ Chunk count stayed 0 across all polls -> chunking likely not triggered by current upload flow.")
    else:
        print("⚠️ Mixed/errored results. Re-run or test with a freshly uploaded document for certainty.")


📄 Scanned docs: 25
📌 Docs with versions: 0

⚠️ Tidak ada dokumen dengan version di tenant ini.
   Upload/seed dokumen melalui flow yang membuat version, lalu jalankan ulang cell ini.
   Kamu bisa lihat ringkasan dokumen di bawah untuk verifikasi cepat.


,doc_id,title,created_at,version_count,latest_version
0,3378c760-a29b-419c-b03e-0c0d55009f21,[1440632] LetteraEsitoRichiedente_1440632_29-0...,2026-03-13T07:14:03.621885Z,0,None
1,de17f179-0da8-43ad-b15d-6c57e2365cb1,[1440632] LetteraEsitoBeneficiario_1440632_29-...,2026-03-13T07:14:03.621885Z,0,None
2,9aa922d0-1de4-43a2-9e70-4800f5150f1b,[1440632] 9. dichiarazione tasso_18-12-2025 17...,2026-03-13T07:14:03.621885Z,0,None
3,3eeac8a4-5ca0-4ada-8b6d-c0631fb6b737,[1440632] 8. Piano di ammortamento_18-12-2025 ...,2026-03-13T07:14:03.621885Z,0,None
4,e0d23be5-99d9-4957-8dbe-e4724b2e6355,[1440632] 7. Erogazione_18-12-2025 17.14.19.pdf,2026-03-13T07:14:03.621885Z,0,None
5,1ceff825-1877-4522-9fa5-f253c3838720,[1440632] 6. Contratto finanziamento_18-12-202...,2026-03-13T07:14:03.621885Z,0,None
6,70b3de40-3dcb-4b5a-b34f-fdc1775593d6,[1440632] 5.1 data della delibera_18-12-2025 1...,2026-03-13T07:14:03.621885Z,0,None
7,cbabeca8-255c-434e-b145-61916504a638,[1440632] 5. delibera banca_18-12-2025 17.13.5...,2026-03-13T07:14:03.621885Z,0,None
8,8fb5a9c5-001f-417d-a58b-ecbcf92e21cd,[1440632] 4. all 4_18-12-2025 17.11.50.pdf,2026-03-13T07:14:03.621885Z,0,None
9,2b2c0c57-e535-4f81-85c0-6d0f30687d94,[1440632] 3. Precisazione del credito_18-12-20...,2026-03-13T07:14:03.621885Z,0,None


### 3.0.1 Smoke Test Upload -> Version -> Chunk

> Gunakan cell ini untuk verifikasi end-to-end apakah flow upload menghasilkan **version** dan **chunk** di KL.
>
> Alur test:
1. Buat dokumen test baru
2. Upload 1 version dengan `raw_text`
3. Poll endpoint chunks selama beberapa percobaan
4. Tampilkan status `PASS` jika chunk muncul, `FAIL` jika tetap 0

In [ ]:
from datetime import datetime, UTC
import time
import pandas as pd
from src.hpvd.adapters.kl_client import DocumentMetadata

# Preconditions
if "client" not in globals():
    raise RuntimeError("client belum terdefinisi. Jalankan Cell 3 dulu.")

# 1) Create a dedicated smoke-test document
ts = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
doc_title = f"SMOKE_CHUNK_TEST_{ts}"

metadata = DocumentMetadata(
    domain="banking",
    action_class="SMOKE_TEST",
    phase_label="ingest_validation",
    tags=["smoke", "chunk", "notebook"],
)

doc = client.create_document(
    title=doc_title,
    document_type="smoke_test",
    metadata=metadata,
    created_by="hpvd-notebook",
)

# 2) Upload version with short raw_text payload
raw_text = (
    "This is an HPVD-KL smoke test document. "
    "Purpose: verify upload creates version and chunk generation pipeline works."
    f" Timestamp={ts}."
)
file_bytes = raw_text.encode("utf-8")

version = client.upload_version(
    document_id=doc.id,
    file_content=file_bytes,
    filename=f"smoke_{ts}.txt",
    raw_text=raw_text,
    uploaded_by="hpvd-notebook",
)

print("✅ Smoke document created")
print(f"   doc_id         : {doc.id}")
print(f"   title          : {doc.title}")
print(f"   version_number : {version.version_number}")

# 3) Poll chunk endpoint
poll_seconds = 5
poll_attempts = 18  # total 90 seconds
poll_rows = []

for attempt in range(1, poll_attempts + 1):
    now = time.strftime("%H:%M:%S")
    try:
        chunk_list = client.list_chunks(doc.id, version.version_number)
        chunk_count = len(chunk_list)
        first_chunk_id = chunk_list[0].chunk_id if chunk_list else None
        status = "ok"
    except Exception as exc:
        chunk_count = -1
        first_chunk_id = None
        status = f"error: {type(exc).__name__}"

    poll_rows.append(
        {
            "attempt": attempt,
            "time": now,
            "chunk_count": chunk_count,
            "first_chunk_id": first_chunk_id,
            "status": status,
        }
    )
    print(f"[{attempt:02d}/{poll_attempts}] {now} chunk_count={chunk_count} status={status}")

    if chunk_count > 0:
        break

    if attempt < poll_attempts:
        time.sleep(poll_seconds)

df_poll = pd.DataFrame(poll_rows)
display(df_poll)

final_count = max([r["chunk_count"] for r in poll_rows if r["chunk_count"] >= 0], default=0)
if final_count > 0:
    print("\n✅ PASS: chunk endpoint returns chunks for uploaded version.")
else:
    print("\n❌ FAIL: version created but chunk count remained 0 during polling window.")
    print("   Indikasi: chunk worker belum aktif / belum ter-trigger untuk flow upload ini.")

C:\Users\junpi\AppData\Local\Temp\ipykernel_20428\3245955880.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


✅ Smoke document created
   doc_id         : a43d4ed2-030c-4d53-9e8a-4ff6780337d2
   title          : SMOKE_CHUNK_TEST_20260316_063817
   version_number : 1
[01/18] 13:38:24 chunk_count=1 status=ok


,attempt,time,chunk_count,first_chunk_id,status
0,1,13:38:24,1,c0001,ok



✅ PASS: chunk endpoint returns chunks for uploaded version.


### 3.1 Load from Snapshot — G1 Resolved

Jika pipeline menerima `pinset_snapshot_id` dari J13, loader bisa me-resolve ke set dokumen yang di-pin secara immutable.

```
GET /snapshots/{snapshot_id}  →  items[]  →  fetch chunks per item  →  DocumentChunk[]
```

Setiap chunk akan memiliki metadata `snapshot_id`, `ontology_version`, dan `calibration_model_version`.


In [21]:
# List available snapshots
snapshots = client.list_snapshots(limit=10)
print(f"📌 Snapshots available: {len(snapshots)}")
for s in snapshots:
    print(f"  {s.get('snapshot_id', '?'):30s}  ontology={s.get('ontology_version', '?')}  calib={s.get('calibration_model_version', '?')}")

# To load from a specific snapshot:
# SNAPSHOT_ID = "PINSET_2026W10"  # e.g. from J13.pinset_snapshot_id
# chunks_snap = loader.load_from_snapshot(SNAPSHOT_ID)
# print(f"Loaded {len(chunks_snap)} chunks from snapshot '{SNAPSHOT_ID}'")
# for c in chunks_snap[:3]:
#     print(f"  {c.chunk_id}  snap={c.metadata.get('snapshot_id')}  ontology={c.metadata.get('ontology_version')}")


📌 Snapshots available: 0


### 3.2 Load with Metadata Filters — G3 + G6 Resolved

Gunakan `load_with_search()` untuk memfilter dokumen berdasarkan `domain`, `action_class`, `phase_label`, atau `tag` sebelum fetching chunks. Memanggil `POST /documents/search` di KL — efisien, tidak perlu fetch semua dokumen.


In [22]:
# Load only credit-analysis documents using server-side metadata filter (G6)
chunks_credit = loader.load_with_search(
    domain="banking",
    action_class="LOAN_EXECUTION",   # structured metadata filter (G3)
    limit=50,
)
print(f"🔍 Filtered load (domain=banking, action_class=LOAN_EXECUTION): {len(chunks_credit)} chunks")

# Also try with a temporal preset (G7 partial — if KL supports it)
chunks_recent = loader.load_with_search(
    preset="LAST_365_DAYS",
    limit=50,
)
print(f"🕐 Temporal filter (LAST_365_DAYS): {len(chunks_recent)} chunks")

# Show breakdown
if chunks_credit:
    df_filtered = pd.DataFrame([
        {
            "chunk_id": c.chunk_id[:22] + "...",
            "topic": c.topic,
            "action_class": c.metadata.get("action_class", ""),
            "phase_label": c.metadata.get("phase_label", ""),
            "text_preview": c.text[:55],
        }
        for c in chunks_credit[:8]
    ])
    display(df_filtered)


🔍 Filtered load (domain=banking, action_class=LOAN_EXECUTION): 0 chunks
🕐 Temporal filter (LAST_365_DAYS): 30 chunks


## 4. Build HPVD Index & Run Pipeline

Kita akan menjalankan pipeline HPVD lengkap:
1. **Build index** dari DocumentChunks (embedding + FAISS)
2. **Construct J13** (PostCoreQuery) — simulasi query dari Serving Adapter
3. **Run pipeline** → produce J14, J15, J16

In [23]:
from src.hpvd.adapters.strategies.document_strategy import (
    DocumentRetrievalStrategy, 
    DocumentRetrievalConfig,
)
from src.hpvd.adapters.pipeline_engine import HPVDPipelineEngine

# Build index
strategy = DocumentRetrievalStrategy(
    DocumentRetrievalConfig(min_similarity=0.0)
)
strategy.build_index(chunks)
print(f"✅ FAISS index built with {len(chunks)} vectors")

# Create pipeline
pipeline = HPVDPipelineEngine(strategies=[strategy])
print(f"✅ Pipeline ready (registered domains: document)")

d:\project\M22\HPVD-M22\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 943.62it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS index built with 36 vectors
✅ Pipeline ready (registered domains: document)


### 4.1 Construct J13 — PostCoreQuery

Ini adalah input query yang dalam pipeline penuh akan datang dari **Serving Adapter (Stage 11)**.

In [24]:
import json

j13_query = {
    "schema_id": "manithy.post_core_query.v2",   # ← updated schema version
    "query_id": "Q_LOAN_SUPPORT_001",
    "scope": {
        "domain": "banking",
        "action_class": "LOAN_EXECUTION",
    },
    # Optional: if set, pipeline resolves knowledge from this pinned snapshot (G1)
    # "pinset_snapshot_id": "PINSET_2026W10",
    "allowed_topics": [],          # no topic filter = search all
    "allowed_corpora": [],         # named corpora filter (new in v2)
    "allowed_doc_types": [],
    "query_payload": {
        "text": "credit risk assessment loan default payment",
    },
}

print("📥 J13 — PostCoreQuery (schema v2):")
print(json.dumps(j13_query, indent=2))


📥 J13 — PostCoreQuery (schema v2):
{
  "schema_id": "manithy.post_core_query.v2",
  "query_id": "Q_LOAN_SUPPORT_001",
  "scope": {
    "domain": "banking",
    "action_class": "LOAN_EXECUTION"
  },
  "allowed_topics": [],
  "allowed_corpora": [],
  "allowed_doc_types": [],
  "query_payload": {
    "text": "credit risk assessment loan default payment"
  }
}


### 4.2 Execute Pipeline → J14, J15, J16

In [25]:
# Run the pipeline
output = pipeline.process_query(j13_query, k=10)

print(f"✅ Pipeline complete!")
print(f"   J14 candidates : {len(output.j14.candidates)}")
print(f"   J15 accepted   : {len(output.j15.accepted)}")
print(f"   J15 rejected   : {len(output.j15.rejected)}")
print(f"   J16 families   : {output.j16.total_families}")
print(f"   J16 members    : {output.j16.total_members}")

✅ Pipeline complete!
   J14 candidates : 10
   J15 accepted   : 10
   J15 rejected   : 0
   J16 families   : 1
   J16 members    : 10


## 5. Inspect J14 — HPVD Retrieval Raw

Hasil retrieval mentah: daftar kandidat analog dengan **calibrated similarity score**.

In [26]:
print("📤 J14 — HPVD_RetrievalRaw")
print(f"   schema_id : {output.j14.schema_id}")
print(f"   query_id  : {output.j14.query_id}")
print(f"   domain    : {output.j14.domain}")
print(f"   candidates: {len(output.j14.candidates)}")
print()

# Show candidates as table
df_j14 = pd.DataFrame([
    {
        "rank": i + 1,
        "candidate_id": c["candidate_id"][:20] + "...",
        "score": round(c["score"], 4),
        "topic": c.get("metadata", {}).get("topic", ""),
        "doc_type": c.get("metadata", {}).get("doc_type", ""),
        "text_preview": c.get("metadata", {}).get("text_preview", "")[:50],
    }
    for i, c in enumerate(output.j14.candidates)
])
df_j14

📤 J14 — HPVD_RetrievalRaw
   schema_id : manithy.hpvd_retrieval_raw.v1
   query_id  : Q_LOAN_SUPPORT_001
   domain    : document
   candidates: 10



,rank,candidate_id,score,topic,doc_type,text_preview
0,1,kl_86f76ccf-72e1-46d...,0.4842,finance,CREDIT_REPORT,[1437613] 11. CR 03 2020_17-11-2025 12.50.10.p...
1,2,kl_43552ca2-1bab-412...,0.4144,finance,CREDIT_REPORT,[1440632] 11. Prima Info CR MARZO 2020_18-12-2...
2,3,kl_78ddd173-eb90-4bc...,0.3610,finance,CREDIT_SPECIFICATION,[1437613] 3. precisazione del credito_17-11-20...
3,4,kl_ecb58759-42c4-4f7...,0.3437,finance,RISK_DECLARATION,[1437613] 9. Dichiarazione Rischio di Credito_...
4,5,kl_2b2c0c57-e535-4f8...,0.3429,finance,CREDIT_SPECIFICATION,[1440632] 3. Precisazione del credito_18-12-20...
5,6,kl_65e28732-3c48-4e9...,0.3312,finance,INTIMAZIONE,[1437613] 1. intimazione_17-11-2025 12.47.20.p...
6,7,kl_c7f32add-474f-4f1...,0.3306,finance,PEC_RECEIPT,[1437613] 2. consegna pec_17-11-2025 12.47.27....
7,8,kl_70b3de40-3dcb-4b5...,0.3270,finance,DELIBERA,[1440632] 5.1 data della delibera_18-12-2025 1...
8,9,kl_7941fc15-ff43-4b5...,0.3125,finance,AMMORTAMENTO,[1437613] 8. piano avvio proced_17-11-2025 12....
9,10,kl_7b682f8d-baac-4c3...,0.3088,finance,PEC_RECEIPT,[1440632] 2. Ricevuta di consegna della PEC_18...


In [27]:
# Full J14 JSON
print(json.dumps(output.j14.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.hpvd_retrieval_raw.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "domain": "document",
  "candidates": [
    {
      "candidate_id": "kl_86f76ccf-72e1-46d6-8fd5-647e7c3e24a0",
      "score": 0.48419585824012756,
      "metadata": {
        "topic": "finance",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: RISK_ASSESSMENT"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_43552ca2-1bab-4125-875a-978a05422abd",
      "score": 0.4143950939178467,
      "metadata": {
        "topic": "finance",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1440632] 11. Prima Info CR MARZO 2020_18-12-2025 17.09.17.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: R"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_78ddd173-eb90-4bc0-ae95-b221a908d599",
      "score": 0.3609911

## 6. Inspect J15 — Phase Filtered Set

Kandidat yang sudah difilter berdasarkan **phase/topic consistency**.

In [28]:
print("📤 J15 — PhaseFilteredSet")
print(f"   schema_id      : {output.j15.schema_id}")
print(f"   accepted count : {len(output.j15.accepted)}")
print(f"   rejected count : {len(output.j15.rejected)}")
print(f"   filter_criteria: {output.j15.filter_criteria}")
print()

# Show accepted candidates
if output.j15.accepted:
    df_j15 = pd.DataFrame([
        {
            "candidate_id": c["candidate_id"][:20] + "...",
            "score": round(c["score"], 4),
            "status": "ACCEPTED",
        }
        for c in output.j15.accepted
    ])
    display(df_j15)

if output.j15.rejected:
    print(f"\nRejected ({len(output.j15.rejected)} candidates):")
    for r in output.j15.rejected:
        print(f"  {r['candidate_id'][:20]}... — reason: {r.get('reason', 'N/A')}")

📤 J15 — PhaseFilteredSet
   schema_id      : manithy.phase_filtered_set.v1
   accepted count : 10
   rejected count : 0
   filter_criteria: {'allowed_topics': [], 'allowed_doc_types': []}



,candidate_id,score,status
0,kl_86f76ccf-72e1-46d...,0.4842,ACCEPTED
1,kl_43552ca2-1bab-412...,0.4144,ACCEPTED
2,kl_78ddd173-eb90-4bc...,0.3610,ACCEPTED
3,kl_ecb58759-42c4-4f7...,0.3437,ACCEPTED
4,kl_2b2c0c57-e535-4f8...,0.3429,ACCEPTED
5,kl_65e28732-3c48-4e9...,0.3312,ACCEPTED
6,kl_c7f32add-474f-4f1...,0.3306,ACCEPTED
7,kl_70b3de40-3dcb-4b5...,0.3270,ACCEPTED
8,kl_7941fc15-ff43-4b5...,0.3125,ACCEPTED
9,kl_7b682f8d-baac-4c3...,0.3088,ACCEPTED


In [29]:
# Full J15 JSON
print(json.dumps(output.j15.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.phase_filtered_set.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "accepted": [
    {
      "candidate_id": "kl_86f76ccf-72e1-46d6-8fd5-647e7c3e24a0",
      "score": 0.48419585824012756,
      "metadata": {
        "topic": "finance",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: RISK_ASSESSMENT"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_43552ca2-1bab-4125-875a-978a05422abd",
      "score": 0.4143950939178467,
      "metadata": {
        "topic": "finance",
        "doc_type": "CREDIT_REPORT",
        "text_preview": "[1440632] 11. Prima Info CR MARZO 2020_18-12-2025 17.09.17.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: R"
      },
      "source_domain": "document"
    },
    {
      "candidate_id": "kl_78ddd173-eb90-4bc0-ae95-b221a908d599",
      "score": 0.3609911799430847,
      "metadata

## 7. Inspect J16 — Analog Family Assignment

Kandidat yang sudah dikelompokkan menjadi **analog families** dengan:
- `coherence` — seberapa koheren family tersebut
- `structural_signature` — ciri struktural family
- `uncertainty_flags` — apakah ada ketidakpastian

In [30]:
print("📤 J16 — AnalogFamilyAssignment")
print(f"   schema_id     : {output.j16.schema_id}")
print(f"   total_families: {output.j16.total_families}")
print(f"   total_members : {output.j16.total_members}")
print()

for fam in output.j16.families:
    print(f"{'='*60}")
    print(f"Family: {fam['family_id']}")
    print(f"  Phase/Topic    : {fam['structural_signature']['phase']}")
    print(f"  Members        : {fam['coherence']['size']}")
    print(f"  Mean confidence: {fam['coherence']['mean_confidence']:.4f}")
    print(f"  Dispersion     : {fam['coherence']['dispersion']:.4f}")
    print(f"  Weak support   : {fam['uncertainty_flags']['weak_support']}")
    print(f"  Phase boundary : {fam['uncertainty_flags']['phase_boundary']}")
    print(f"  Members:")
    for m in fam['members']:
        print(f"    - {m['candidate_id'][:25]}...  score={m['score']:.4f}")

📤 J16 — AnalogFamilyAssignment
   schema_id     : manithy.analog_family_assignment.v1
   total_families: 1
   total_members : 10

Family: DF_001
  Phase/Topic    : finance
  Members        : 10
  Mean confidence: 0.3556
  Dispersion     : 0.0514
  Weak support   : True
  Phase boundary : False
  Members:
    - kl_86f76ccf-72e1-46d6-8fd...  score=0.4842
    - kl_43552ca2-1bab-4125-875...  score=0.4144
    - kl_78ddd173-eb90-4bc0-ae9...  score=0.3610
    - kl_ecb58759-42c4-4f7b-83c...  score=0.3437
    - kl_2b2c0c57-e535-4f81-85c...  score=0.3429
    - kl_65e28732-3c48-4e9f-aef...  score=0.3312
    - kl_c7f32add-474f-4f13-93c...  score=0.3306
    - kl_70b3de40-3dcb-4b5a-b34...  score=0.3270
    - kl_7941fc15-ff43-4b58-a3a...  score=0.3125
    - kl_7b682f8d-baac-4c36-901...  score=0.3088


In [31]:
# Family summary table
df_j16 = pd.DataFrame([
    {
        "family_id": f["family_id"],
        "topic": f["structural_signature"]["phase"],
        "members": f["coherence"]["size"],
        "mean_confidence": round(f["coherence"]["mean_confidence"], 4),
        "dispersion": round(f["coherence"]["dispersion"], 4),
        "weak_support": f["uncertainty_flags"]["weak_support"],
    }
    for f in output.j16.families
])
df_j16

,family_id,topic,members,mean_confidence,dispersion,weak_support
0,DF_001,finance,10,0.3556,0.0514,True


In [32]:
# Full J16 JSON
print(json.dumps(output.j16.to_dict(), indent=2, ensure_ascii=False))

{
  "schema_id": "manithy.analog_family_assignment.v1",
  "query_id": "Q_LOAN_SUPPORT_001",
  "families": [
    {
      "family_id": "DF_001",
      "members": [
        {
          "candidate_id": "kl_86f76ccf-72e1-46d6-8fd5-647e7c3e24a0",
          "score": 0.48419585824012756,
          "metadata": {
            "topic": "finance",
            "doc_type": "CREDIT_REPORT",
            "text_preview": "[1437613] 11. CR 03 2020_17-11-2025 12.50.10.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: RISK_ASSESSMENT"
          },
          "source_domain": "document"
        },
        {
          "candidate_id": "kl_43552ca2-1bab-4125-875a-978a05422abd",
          "score": 0.4143950939178467,
          "metadata": {
            "topic": "finance",
            "doc_type": "CREDIT_REPORT",
            "text_preview": "[1440632] 11. Prima Info CR MARZO 2020_18-12-2025 17.09.17.pdf. Document type: CREDIT_REPORT. Domain: finance. Action: R"
          },
          "source_domain": "do

## 8. Save Results to File

Simpan output lengkap ke JSON file untuk referensi.

In [33]:
import os
from datetime import datetime

# Save full pipeline output
output_dir = "hpvd_outputs/kl_integration"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_dir, f"pipeline_output_{timestamp}.json")

full_output = {
    "metadata": {
        "timestamp": timestamp,
        "kl_url": KL_URL,
        "total_kl_documents": len(documents),
        "total_chunks": len(chunks),
        "kl_managed_chunks": sum(1 for c in chunks if not c.metadata.get("synthetic_chunk")),
        "synthetic_chunks": sum(1 for c in chunks if c.metadata.get("synthetic_chunk")),
        "schema_version": "manithy.post_core_query.v2",
    },
    "j13_query": j13_query,
    "pipeline_output": output.to_dict(),
}

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(full_output, f, indent=2, ensure_ascii=False)

print(f"💾 Output saved to: {output_file}")
print(f"   File size: {os.path.getsize(output_file):,} bytes")


💾 Output saved to: hpvd_outputs/kl_integration\pipeline_output_20260314_203925.json
   File size: 15,689 bytes


## 9. Try Different Queries

Coba ubah query di bawah untuk lihat bagaimana HPVD meresponse query yang berbeda.

In [34]:
# Try different queries!
queries = [
    {"text": "bank deliberation approval decision", "label": "Bank Decision"},
    {"text": "legal notice payment intimation", "label": "Legal Notice"},
    {"text": "fidejussione garanzia guarantee", "label": "Guarantee"},
    {"text": "loan contract finanziamento", "label": "Contract"},
]

results_summary = []
for q in queries:
    j13 = {
        "query_id": f"multi_q_{q['label'].replace(' ', '_').lower()}",
        "scope": {"domain": "banking"},
        "allowed_topics": [],
        "query_payload": {"text": q["text"]},
    }
    out = pipeline.process_query(j13, k=5)
    
    top_candidate = out.j14.candidates[0] if out.j14.candidates else {}
    results_summary.append({
        "query": q["label"],
        "candidates": len(out.j14.candidates),
        "families": out.j16.total_families,
        "top_score": round(top_candidate.get("score", 0), 4),
        "top_topic": top_candidate.get("metadata", {}).get("topic", ""),
    })

df_results = pd.DataFrame(results_summary)
df_results

,query,candidates,families,top_score,top_topic
0,Bank Decision,5,1,0.5519,finance
1,Legal Notice,5,1,0.3813,finance
2,Guarantee,5,1,0.3939,finance
3,Contract,5,1,0.5790,finance


## 10. Metadata-Filtered Query (G3 + G6)

Demonstrasi query dengan kombinasi:
1. **KL server-side filter** via `load_with_search()` — hanya load dokumen yang relevan dari KL
2. **Phase filter** via `allowed_topics` di J13 — J15 emitter reject kandidat di luar topik


In [35]:
# Query with metadata filter: server-side filter via load_with_search (G3 + G6),
# then pass to pipeline with allowed_topics for phase-level filtering.
from src.hpvd.adapters.strategies.document_strategy import (
    DocumentRetrievalStrategy,
    DocumentRetrievalConfig,
)
from src.hpvd.adapters.pipeline_engine import HPVDPipelineEngine

# Build a focused index using KL metadata filter (G6)
chunks_credit_only = loader.load_with_search(
    domain="banking",
    action_class="LOAN_EXECUTION",
    limit=50,
)

focused_strategy = DocumentRetrievalStrategy(
    DocumentRetrievalConfig(min_similarity=0.0)
)
focused_strategy.build_index(chunks_credit_only if chunks_credit_only else chunks)
pipeline_filtered = HPVDPipelineEngine(strategies=[focused_strategy])

# J13 with topic filter
j13_filtered = {
    "schema_id": "manithy.post_core_query.v2",
    "query_id": "Q_FILTERED_CREDIT",
    "scope": {"domain": "banking", "action_class": "LOAN_EXECUTION"},
    "allowed_topics": ["credit_analysis"],   # phase-level filter (J15 stage)
    "allowed_corpora": [],
    "query_payload": {"text": "credit risk assessment"},
}

out_filtered = pipeline_filtered.process_query(j13_filtered, k=10)

print(f"🔍 Filtered query (domain=banking / topic=credit_analysis):")
print(f"   Index size : {len(chunks_credit_only) if chunks_credit_only else len(chunks)} chunks (KL metadata filter)")
print(f"   Candidates : {len(out_filtered.j14.candidates)}")
print(f"   Families   : {out_filtered.j16.total_families}")
print()

for c in out_filtered.j14.candidates:
    meta = c.get("metadata", {})
    print(f"  score={c['score']:.4f}  topic={meta.get('topic', '')}  action={meta.get('action_class', '')}  {meta.get('text_preview', '')[:45]}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1073.33it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Filtered query (domain=banking / topic=credit_analysis):
   Index size : 36 chunks (KL metadata filter)
   Candidates : 0
   Families   : 0



---

## Summary

| Stage | Output | Description |
|-------|--------|-------------|
| KL API | Documents | Banking documents via `GET /documents` (auth via API key) |
| G1 | Snapshot | Immutable pinset resolved via `GET /snapshots/{id}` |
| G2 | KL-managed chunks | Real chunk content per document version (`GET .../chunks`) |
| G3 | Structured metadata | `domain`, `action_class`, `phase_label`, `tags` on docs & chunks |
| G6 | Server-side search | Metadata-filtered fetch via `POST /documents/search` |
| Loader | DocumentChunks | Auto-selects KL chunks → fallback synthetic if none |
| J13 v2 | PostCoreQuery | `schema_id: manithy.post_core_query.v2`, `pinset_snapshot_id` support |
| J14 | RetrievalRaw | Ranked candidates with similarity scores |
| J15 | PhaseFilteredSet | Accepted/rejected by phase filter |
| J16 | AnalogFamilyAssignment | Grouped into families with coherence metrics |
